# Day 11 - Softmax and Reductions in CuTe DSL

This day implements stable row-wise softmax and compares CuTe's explicit
reduction hierarchy with Triton's program-level reductions.

```text
max_i x_i
e_i = exp(x_i - max)
sum_i e_i
y_i = e_i / sum
```

Subtracting the row maximum is required for numerical stability. Without it,
large positive inputs can overflow `exp`, while very negative inputs can
underflow every numerator to zero.

## Reduction has two levels

The official CuTe TensorSSA notebook introduces:

```python
local_max = values.reduce(
    cute.ReductionOp.MAX,
    -cutlass.Float32.inf,
    reduction_profile=0,
)
```

`TensorSSA.reduce` only combines values already owned by one thread. It lowers
a register-resident vector reduction; it does not communicate with other
lanes. A row distributed across a warp therefore needs a second level:

```text
thread register values
    | TensorSSA.reduce
thread partial
    | shuffle_sync_bfly at offsets 16, 8, 4, 2, 1
warp result replicated to all lanes
```

This lesson uses one 32-thread warp per row. Lane `l` owns columns:

```text
l, l + 32, l + 64, ...
```

At each item index, adjacent lanes access adjacent columns, preserving
coalesced global-memory traffic. An irregular row such as 1003 columns gives
each lane 32 register slots; slots beyond column 1002 are initialized to
negative infinity. They cannot affect the maximum, and `exp(-inf)` contributes
zero to the sum.

## Challenge

Implement three pieces in `puzzle.py`:

1. `warp_reduce_max`: five butterfly shuffles combined with `cute.arch.fmax`.
2. `warp_reduce_sum`: the same shuffle tree combined with addition.
3. `WarpSoftmax.kernel`: load the local register fragment, perform local and
   warp max reductions, exponentiate, perform local and warp sum reductions,
   normalize, and store valid columns.

The supplied tests cover:

- one value and widths smaller than one warp;
- exact powers of two;
- multiple values per lane;
- irregular width 1003;
- large positive and negative logits;
- row sums equal to one.

Run the standalone files:

```bash
python day11_softmax_reduction/puzzle.py
python day11_softmax_reduction/solution.py
python day11_softmax_reduction/solution.py --benchmark
```

## Why one warp per row?

CuTe DSL 4.6 exposes register reductions and warp shuffle primitives, but no
single generic CTA-wide reduction call. A multi-warp row would need:

1. one local reduction per thread;
2. one reduction per warp;
3. lane 0 of each warp writing a shared-memory partial;
4. a CTA barrier;
5. a final warp reducing those partials;
6. shared-memory broadcast back to every warp.

That is a useful follow-up, but it obscures the core reduction semantics. The
one-warp design is correct for widths up to this lesson's 4096-column limit
and makes the hierarchy visible without shared-memory synchronization.

The tradeoff is register pressure: width 1024 means 32 values per lane. Triton
may choose more warps and a different lowering for the same row, so benchmark
results are part of the lesson rather than an assumed conclusion.

## CuTe versus Triton

The notebook contains a matched Triton kernel and benchmarks both
implementations with `triton.testing.do_bench`.

| Concern | CuTe Day 11 | Triton Day 13 |
| --- | --- | --- |
| Thread ownership | Explicit lane-strided register tensor | Logical vector from `tl.arange` |
| Local reduction | `TensorSSA.reduce` | Hidden inside `tl.max` / `tl.sum` lowering |
| Cross-lane reduction | Explicit butterfly shuffles | Compiler-generated |
| Padding identity | Register slots initialized to `-inf` | Masked load uses `other=-inf` |
| Launch mapping | Exactly one warp per row | One program per row, configurable warps |
| Specialization | `WarpSoftmax(n_cols)` | `BLOCK_SIZE: tl.constexpr` |

The comparison uses the same FP32 input tensors and shapes:
128, 512, 1003, and 1024 columns. Effective bandwidth counts one input read
and one output write:

```text
GB/s = 2 * rows * columns * sizeof(float) / elapsed_seconds / 1e9
```

This is a traffic proxy, not a claim that softmax performs only memory work.
The exponential and two reductions also consume compute and synchronization
resources.

## API notes

- `reduction_profile=0` reduces the only mode of the 1-D TensorSSA.
- Identities must match the operation: `-inf` for maximum, zero for addition.
- Butterfly reduction works because the group size is a power of two and all
  32 lanes participate.
- Every lane receives the final value, so scalar subtraction and division
  broadcast naturally across the local TensorSSA.
- Shape is specialized at compile time because register tensor dimensions
  must be static.

The reduction pattern follows NVIDIA CUTLASS's official CuTe DSL TensorSSA
notebook and the warp-shuffle reduction used in the official Ampere Flash
Attention example.

## Reduction checkpoint: local values are not warp values

A lane first owns a static register tensor. Calling
`TensorSSA.reduce` combines only that tensor:

```text
lane 0: [x0, x32, x64, ...] -> partial_0
lane 1: [x1, x33, x65, ...] -> partial_1
...
lane31: [x31, x63, x95, ...] -> partial_31
```

The five `shuffle_sync_bfly` steps then exchange partials at XOR
distances 16, 8, 4, 2, and 1. After the fifth step, every lane has
the same row-wide result. The kernel performs this hierarchy twice:
first for the maximum and then for the exponential sum.

In [ ]:
def butterfly_partners(lane):
    return [lane ^ offset for offset in (16, 8, 4, 2, 1)]


for lane in (0, 1, 17, 31):
    print(f"lane {lane:2d} exchanges with {butterfly_partners(lane)}")

## Challenge implementation

Complete TODO(1)-TODO(3). The host wrapper specializes
`values_per_lane = ceil(n_cols / 32)`, so the register tensor has a
static shape. Padding uses the correct reduction identities:
negative infinity for max and, after exponentiation, zero for sum.

In [ ]:
#!/usr/bin/env python3
# SPDX-License-Identifier: BSD-3-Clause
"""Day 11 - implement row-wise softmax with local and warp reductions."""

from __future__ import annotations

import argparse

import cutlass
import cutlass.cute as cute
import cutlass.cute.testing as testing
from cutlass.cute.runtime import from_dlpack


WARP_SIZE = 32


def warp_reduce_max(value: cutlass.Float32) -> cutlass.Float32:
    # TODO(1): combine all 32 lanes with shuffle_sync_bfly and cute.arch.fmax.
    raise NotImplementedError("Day 11 TODO(1): warp max reduction")


def warp_reduce_sum(value: cutlass.Float32) -> cutlass.Float32:
    # TODO(2): combine all 32 lanes with shuffle_sync_bfly and addition.
    raise NotImplementedError("Day 11 TODO(2): warp sum reduction")


class WarpSoftmax:
    """Compile-time-specialized one-warp-per-row softmax."""

    def __init__(self, n_cols: int) -> None:
        if n_cols <= 0:
            raise ValueError("n_cols must be positive")
        self.n_cols = n_cols
        self.values_per_lane = (n_cols + WARP_SIZE - 1) // WARP_SIZE

    @cute.jit
    def __call__(self, m_x: cute.Tensor, m_y: cute.Tensor) -> None:
        if cutlass.const_expr(
            m_x.element_type != cutlass.Float32
            or m_y.element_type != cutlass.Float32
        ):
            raise TypeError("Day 11 supports Float32 tensors")
        self.kernel(m_x, m_y).launch(
            grid=(m_x.shape[0], 1, 1),
            block=(WARP_SIZE, 1, 1),
        )

    @cute.kernel
    def kernel(self, g_x: cute.Tensor, g_y: cute.Tensor) -> None:
        lane, _, _ = cute.arch.thread_idx()
        row, _, _ = cute.arch.block_idx()

        # TODO(3):
        # 1. Create a Float32 register tensor with self.values_per_lane values.
        # 2. Load columns lane + item * 32; pad OOB entries with -infinity.
        # 3. Use TensorSSA.reduce(MAX), then warp_reduce_max.
        # 4. Exponentiate values - row_max.
        # 5. Use TensorSSA.reduce(ADD), then warp_reduce_sum.
        # 6. Normalize and store only in-bounds columns.
        raise NotImplementedError("Day 11 TODO(3): stable row softmax")


def _validate_input(x) -> None:
    import torch

    if not x.is_cuda:
        raise ValueError("x must be a CUDA tensor")
    if x.dtype != torch.float32:
        raise TypeError("x must have dtype torch.float32")
    if x.ndim != 2 or not x.is_contiguous():
        raise ValueError("x must be a contiguous 2-D tensor")
    if x.shape[0] == 0 or x.shape[1] == 0:
        raise ValueError("x dimensions must be non-zero")
    if x.shape[1] > 4096:
        raise ValueError("this one-warp lesson supports at most 4096 columns")


def prepare_softmax(x):
    import torch

    _validate_input(x)
    y = torch.empty_like(x)
    x_cute = from_dlpack(x, assumed_align=4)
    y_cute = from_dlpack(y, assumed_align=4)
    compiled = cute.compile(WarpSoftmax(x.shape[1]), x_cute, y_cute)
    return y, compiled, x_cute, y_cute


def softmax(x):
    y, compiled, x_cute, y_cute = prepare_softmax(x)
    compiled(x_cute, y_cute)
    return y


def test_softmax() -> None:
    import torch

    torch.manual_seed(20260915)
    for shape in ((1, 1), (7, 31), (33, 128), (65, 512), (257, 1003)):
        x = torch.randn(shape, device="cuda", dtype=torch.float32) * 7.0
        actual = softmax(x)
        expected = torch.softmax(x, dim=1)
        torch.testing.assert_close(actual, expected, atol=2e-6, rtol=2e-5)
        torch.testing.assert_close(
            actual.sum(dim=1),
            torch.ones(shape[0], device="cuda"),
            atol=2e-6,
            rtol=2e-6,
        )

    extreme = torch.tensor(
        [[-1000.0, 0.0, 1000.0], [1000.0, 1000.0, 999.0]],
        device="cuda",
    )
    torch.testing.assert_close(
        softmax(extreme),
        torch.softmax(extreme, dim=1),
        atol=2e-6,
        rtol=2e-5,
    )
    print("Day 11 CuTe softmax: correctness OK")


def benchmark_softmax(
    rows: int,
    columns: int,
    warmup: int = 25,
    iterations: int = 100,
) -> tuple[float, float]:
    import torch

    x = torch.randn(rows, columns, device="cuda", dtype=torch.float32)
    y, compiled, x_cute, y_cute = prepare_softmax(x)
    compiled(x_cute, y_cute)
    torch.cuda.synchronize()
    average_us = testing.benchmark(
        compiled,
        kernel_arguments=testing.JitArguments(x_cute, y_cute),
        warmup_iterations=warmup,
        iterations=iterations,
    )
    bytes_moved = 2 * x.numel() * x.element_size()
    return average_us, bytes_moved / (average_us * 1_000)


def benchmark_suite(rows: int = 4096) -> None:
    print("columns | CuTe us | effective GB/s")
    print("------- | ------- | --------------")
    for columns in (128, 512, 1003, 1024):
        average_us, bandwidth = benchmark_softmax(rows, columns)
        print(f"{columns:7d} | {average_us:7.2f} | {bandwidth:14.2f}")


def run(rows: int, benchmark: bool) -> None:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Day 11 needs a CUDA device")
    cutlass.cuda.initialize_cuda_context()
    test_softmax()
    if benchmark:
        benchmark_suite(rows)

## Correctness tests

These tests include non-power-of-two widths and logits near
floating-point overflow. Passing ordinary random values is not
enough: the extreme test catches a missing max subtraction, and
the row-sum assertion catches incorrect reduction scope.

In [ ]:
test_softmax()

## CuTe-only benchmark

This compiles once per shape and measures only repeated compiled
launches. Effective bandwidth counts one FP32 read and one FP32
write. Run it after correctness passes.

In [ ]:
benchmark_suite(rows=4096)

## Matched Triton comparison

Triton expresses the same stable algorithm with `tl.max` and
`tl.sum`. The compiler owns the lane-level reduction strategy. The
cell below uses the same tensors, output buffers, warmup,
repetitions, and `triton.testing.do_bench` timing path for both
kernels. This avoids comparing numbers gathered under different
benchmark harnesses.

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def triton_softmax_kernel(
    x_ptr, y_ptr, n_cols, BLOCK_SIZE: tl.constexpr
):
    row = tl.program_id(0)
    columns = tl.arange(0, BLOCK_SIZE)
    mask = columns < n_cols
    values = tl.load(
        x_ptr + row * n_cols + columns,
        mask=mask,
        other=-float("inf"),
    )
    values -= tl.max(values, axis=0)
    numerators = tl.exp(values)
    denominator = tl.sum(numerators, axis=0)
    tl.store(
        y_ptr + row * n_cols + columns,
        numerators / denominator,
        mask=mask,
    )


def launch_triton_softmax(x, y):
    rows, columns = x.shape
    block_size = triton.next_power_of_2(columns)
    num_warps = min(max(block_size // 256, 1), 8)
    triton_softmax_kernel[(rows,)](
        x,
        y,
        columns,
        BLOCK_SIZE=block_size,
        num_warps=num_warps,
    )

In [ ]:
def compare_cute_and_triton(rows=4096, warmup=25, repetitions=100):
    print("columns | CuTe us | Triton us | CuTe GB/s | Triton GB/s")
    print("------- | ------- | --------- | --------- | -----------")
    for columns in (128, 512, 1003, 1024):
        x = torch.randn(
            rows, columns, device="cuda", dtype=torch.float32
        )
        cute_y, compiled, x_cute, y_cute = prepare_softmax(x)
        triton_y = torch.empty_like(x)
        compiled(x_cute, y_cute)
        launch_triton_softmax(x, triton_y)
        torch.cuda.synchronize()
        reference = torch.softmax(x, dim=1)
        torch.testing.assert_close(
            cute_y, reference, atol=2e-6, rtol=2e-5
        )
        torch.testing.assert_close(
            triton_y, reference, atol=2e-6, rtol=2e-5
        )

        cute_ms = triton.testing.do_bench(
            lambda: compiled(x_cute, y_cute),
            warmup=warmup,
            rep=repetitions,
        )
        triton_ms = triton.testing.do_bench(
            lambda: launch_triton_softmax(x, triton_y),
            warmup=warmup,
            rep=repetitions,
        )
        bytes_moved = 2 * x.numel() * x.element_size()
        cute_gbps = bytes_moved / (cute_ms * 1e6)
        triton_gbps = bytes_moved / (triton_ms * 1e6)
        print(
            f"{columns:7d} | {cute_ms * 1000:7.2f} | "
            f"{triton_ms * 1000:9.2f} | {cute_gbps:9.2f} | "
            f"{triton_gbps:11.2f}"
        )


compare_cute_and_triton()

## Interpret the result

Do not reduce the comparison to language syntax. Check how the
gap changes with width:

- Small rows emphasize launch and scheduling overhead.
- Wider rows increase CuTe's per-lane register fragment.
- Width 1003 measures the cost of padded work and predicates.
- Triton's `num_warps` changes at larger power-of-two blocks, while
  this CuTe kernel deliberately remains one warp per row.

A production follow-up would tune warps per row, vectorize global
accesses, and implement a shared-memory cross-warp reduction.